# Import required libraries and mount Google Drive

In [1]:
# notebooks/01_Data_Preparation.ipynb

import os
import zipfile
import json
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
from google.colab import drive
from tqdm import tqdm

# Mount Google Drive to access persistent storage
drive.mount('/content/drive')

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)
import src.config as config

Mounted at /content/drive


# Extract the RAW dataset zip file to local storage

In [2]:
print("Extracting raw dataset zip file to local runtime storage...")
os.makedirs(config.DATA_ROOT_DIR_RAW, exist_ok=True)

with zipfile.ZipFile(config.ZIP_SOURCE_PATH_RAW, 'r') as zip_ref:
    unzip_targets = zip_ref.namelist()
    for file in tqdm(unzip_targets, desc="Extracting raw data"):
        zip_ref.extract(file, config.LOCAL_EXTRACT_DIR)

print(f"Extraction complete. Raw data directory: {config.DATA_ROOT_DIR_RAW}")

Extracting raw dataset zip file to local runtime storage...


Extracting raw data: 100%|██████████| 1847/1847 [06:01<00:00,  5.11it/s]

Extraction complete. Raw data directory: /content/MICCAI_BraTS2020_TrainingData


In [3]:
# Handle sample 355 misnamed segmentation file
sample_355_dir = os.path.join(config.DATA_ROOT_DIR_RAW, "BraTS20_Training_355")
old_seg_path = os.path.join(sample_355_dir, "W39_1998.09.19_Segm.nii")
new_seg_path = os.path.join(sample_355_dir, "BraTS20_Training_355_seg.nii")

if os.path.exists(old_seg_path):
    os.rename(old_seg_path, new_seg_path)
    print("Successfully renamed sample 355 segmentation file to BraTS20_Training_355_seg.nii")

Successfully renamed sample 355 segmentation file to BraTS20_Training_355_seg.nii


# Handle sample 355 misnamed segmentation file
sample_355_dir = os.path.join(config.DATA_ROOT_DIR_RAW, "BraTS20_Training_355")
old_seg_path = os.path.join(sample_355_dir, "W39_1998.09.19_Segm.nii")
new_seg_path = os.path.join(sample_355_dir, "BraTS20_Training_355_seg.nii")

if os.path.exists(old_seg_path):
    os.rename(old_seg_path, new_seg_path)
    print("Successfully renamed sample 355 segmentation file to BraTS20_Training_355_seg.nii")

# Isotropic Bounding Box Crop, Resize to 128^3, and Save/Zip to Drive

In [4]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.6 MB/s eta 0:00:00


In [5]:
import shutil
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F

# Define paths to metadata
raw_name_mapping_csv = os.path.join(config.DATA_ROOT_DIR_RAW, "name_mapping.csv")
raw_survival_info_csv = os.path.join(config.DATA_ROOT_DIR_RAW, "survival_info.csv")

df_meta = pd.read_csv(raw_name_mapping_csv)
subject_ids = df_meta['BraTS_2020_subject_ID'].dropna().tolist()

# -------------------------------------------------------------------------
# PHASE 1: Find the largest brain dimension across the whole dataset
# -------------------------------------------------------------------------
print("Phase 1: Scanning dataset to determine global maximum brain bounding box dimensions (a, b, c)...")
max_spans = [0, 0, 0]

for subject_id in tqdm(subject_ids, desc="  Scanning for global a, b, c"):
    ref_path = os.path.join(config.DATA_ROOT_DIR_RAW, subject_id, f"{subject_id}_{config.MODALITIES[0]}.nii")
    if not os.path.exists(ref_path):
        continue

    img = nib.load(ref_path).get_fdata()
    non_zero_coords = np.argwhere(img > 0)
    if len(non_zero_coords) == 0:
        continue

    min_idx = non_zero_coords.min(axis=0)
    max_idx = non_zero_coords.max(axis=0)
    spans = max_idx - min_idx + 1

    for d in range(3):
        if spans[d] > max_spans[d]:
            max_spans[d] = spans[d]

A = int(max_spans[0])
B = int(max_spans[1])
C = int(max_spans[2])
print(f"  Global dimensions identified: a={max_spans[0]}, b={max_spans[1]}, c={max_spans[2]}. Unified bounding box size: {A}x{B}x{C}")

# -------------------------------------------------------------------------
# PHASE 2: Isotropic Bounding Box Crop, Resize, and Metadata Correction
# -------------------------------------------------------------------------
print("\nPhase 2: Executing Bounding Box Crop and Metadata Correction Loop...")
for subject_id in tqdm(subject_ids, desc="  Preprocessing subjects"):
    subj_raw_dir = os.path.join(config.DATA_ROOT_DIR_RAW, subject_id)
    subj_out_dir = os.path.join(config.DATA_ROOT_DIR, subject_id)

    modality_paths = [os.path.join(subj_raw_dir, f"{subject_id}_{mod}.nii") for mod in config.MODALITIES]
    seg_path = os.path.join(subj_raw_dir, f"{subject_id}_seg.nii")

    if not (all(os.path.exists(p) for p in modality_paths) and os.path.exists(seg_path)):
        continue

    os.makedirs(subj_out_dir, exist_ok=True)

    # Load reference sequence header for physical space matrix mappings
    ref_nib = nib.load(modality_paths[0])
    ref_data = ref_nib.get_fdata()
    ref_affine = ref_nib.affine

    non_zero_coords = np.argwhere(ref_data > 0)
    if len(non_zero_coords) == 0:
        continue

    min_idx = non_zero_coords.min(axis=0)
    max_idx = non_zero_coords.max(axis=0)
    center_c = ((min_idx + max_idx) // 2).astype(int)

    box_shape = np.array([A, B, C])
    half_box = box_shape // 2
    start_coords = center_c - half_box
    end_coords = start_coords + box_shape

    all_paths = modality_paths + [seg_path]
    for p in all_paths:
        file_nib = nib.load(p)
        file_data = file_nib.get_fdata()

        # Symmetrically handle matrix bounds overflows
        pad_before = np.maximum(0, -start_coords)
        pad_after = np.maximum(0, end_coords - np.array(file_data.shape))

        if np.any(pad_before > 0) or np.any(pad_after > 0):
            file_data = np.pad(
                file_data,
                ((pad_before[0], pad_after[0]),
                 (pad_before[1], pad_after[1]),
                 (pad_before[2], pad_after[2])),
                mode='constant', constant_values=0
            )
            adj_start = start_coords + pad_before
        else:
            adj_start = start_coords

        cropped_data = file_data[
            adj_start[0]:adj_start[0]+A,
            adj_start[1]:adj_start[1]+B,
            adj_start[2]:adj_start[2]+C
        ]

        # Cast back to the original data type to prevent file size expansion
        cropped_data = cropped_data.astype(file_nib.get_data_dtype())

        # Removed reshaping
        ########################################################################################
        # # Reshape to 128*128*128

        # import numpy as np
        # import scipy.ndimage as ndimage
        # from monai.transforms import Resize

        # is_seg = "_seg.nii" in p

        # if is_seg:
        #     # For ground-truth segmentations: Use MONAI nearest-neighbor to preserve integer labels
        #     tensor_data = torch.from_numpy(cropped_data).float().unsqueeze(0)  # Shape: (C, H, W, D)
        #     seg_resizer = Resize(spatial_size=(128, 128, 128), mode="nearest")
        #     resized_tensor = seg_resizer(tensor_data)
        #     resized_data = resized_tensor.squeeze(0).numpy().astype(np.int32)
        # else:
        #     # For multi-modal structural images: Use SciPy zoom with order=3 for 3D Cubic B-spline
        #     zoom_factors = (128 / cropped_data.shape[0], 128 / cropped_data.shape[1], 128 / cropped_data.shape[2])

        #     # 1. Execute 3D Cubic B-spline zoom
        #     resized_data = ndimage.zoom(cropped_data, zoom=zoom_factors, order=3)

        #     # 2. Clip cubic ringing artifacts (prevents background voxels from dipping below 0)
        #     resized_data = np.clip(resized_data, a_min=0.0, a_max=None)

        #     # 3. Cast to float32 (SciPy outputs float64, which causes PyTorch tensor runtime errors later)
        #     resized_data = resized_data.astype(np.float32)

        ########################################################################################

        # -----------------------------------------------------------------
        # METADATA FIXES: Correct spatial origin
        # -----------------------------------------------------------------
        new_affine = np.copy(ref_affine)

        # Shift origin matrix coordinates to preserve 3D visualization positioning
        new_affine[:3, 3] = ref_affine[:3, :3] @ adj_start + ref_affine[:3, 3]

        out_file_path = os.path.join(subj_out_dir, os.path.basename(p))
        new_nib = nib.Nifti1Image(cropped_data, new_affine)

        # Zooms remain unchanged since images are only cropped, not scaled
        new_nib.header.set_zooms(file_nib.header.get_zooms())

        nib.save(new_nib, out_file_path)

# -------------------------------------------------------------------------
# PHASE 3: Package preprocessed records and transfer back to Drive storage
# -------------------------------------------------------------------------
print("\nPhase 3: Copying Metadata CSV Records to Preprocessed Target Directory...")
shutil.copy(raw_name_mapping_csv, config.NAME_MAPPING_CSV)
if os.path.exists(raw_survival_info_csv):
    shutil.copy(raw_survival_info_csv, config.SURVIVAL_INFO_CSV)

print("\nPhase 4: Compressing and Saving Preprocessed Dataset Zip back to Drive...")
drive_zip_dir = os.path.dirname(config.ZIP_SOURCE_PATH)
os.makedirs(drive_zip_dir, exist_ok=True)
zip_base_name = os.path.splitext(config.ZIP_SOURCE_PATH)[0]
shutil.make_archive(zip_base_name, 'zip', config.DATA_ROOT_DIR)

print(f"  Complete preprocessing cycle finished! Archive saved to drive at: {config.ZIP_SOURCE_PATH}")

Phase 1: Scanning dataset to determine global maximum brain bounding box dimensions (a, b, c)...


  Scanning for global a, b, c: 100%|██████████| 369/369 [01:48<00:00,  3.41it/s]


  Global dimensions identified: a=155, b=193, c=149. Unified bounding box size: 155x193x149

Phase 2: Executing Bounding Box Crop and Metadata Correction Loop...


  Preprocessing subjects: 100%|██████████| 369/369 [06:53<00:00,  1.12s/it]



Phase 3: Copying Metadata CSV Records to Preprocessed Target Directory...

Phase 4: Compressing and Saving Preprocessed Dataset Zip back to Drive...
  Complete preprocessing cycle finished! Archive saved to drive at: /content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip


# Parse predefined text splits and generate JSON configuration

In [6]:
print("\nParsing custom split text files and generating JSON records...")

def load_subject_ids_from_txt(txt_filename):
    """Helper function to read subject IDs from a text file and strip HG_/LG_ prefixes."""
    txt_path = os.path.join(PROJECT_ROOT, "data", txt_filename)
    if not os.path.exists(txt_path):
        print(f"Error: Could not find {txt_path}")
        return []

    cleaned_ids = []
    with open(txt_path, 'r') as f:
        for line in f:
            raw_id = line.strip()
            if not raw_id:
                continue

            # Remove HG_ or LG_ prefix to match dataset directory structure
            if raw_id.startswith("HG_") or raw_id.startswith("LG_"):
                cleaned_id = raw_id[3:]
            else:
                cleaned_id = raw_id

            cleaned_ids.append(cleaned_id)

    return cleaned_ids

def create_patient_records(subject_ids):
    """Helper function to generate the image/label dictionary for a list of subjects."""
    records = []
    for subject_id in subject_ids:
        # Paths for each structural modality matching the specified sequence order
        modality_paths = [
            os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_{mod}.nii")
            for mod in config.MODALITIES
        ]

        # Path for the segmentation target mask
        seg_path = os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_seg.nii")

        # Check all 5 files exist before adding to records
        files_exist = all(os.path.exists(p) for p in modality_paths) and os.path.exists(seg_path)
        if not files_exist:
            print(f"Warning: Missing files for subject {subject_id}. Skipping.")
            continue

        records.append({
            "image": modality_paths,
            "label": seg_path
        })
    return records


Parsing custom split text files and generating JSON records...


In [7]:
# 1. Read IDs from the text files
train_ids = load_subject_ids_from_txt("train.txt")
val_ids = load_subject_ids_from_txt("val.txt")
test_ids = load_subject_ids_from_txt("test.txt")

# 2. Build the dictionaries linking files
train_records = create_patient_records(train_ids)
val_records = create_patient_records(val_ids)
test_records = create_patient_records(test_ids)

print(f"Split distribution summary:")
print(f"Train samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")
print(f"Test samples: {len(test_records)}")

Split distribution summary:
Train samples: 219
Validation samples: 50
Test samples: 100


# Save dataset split configuration to a single JSON file

In [8]:
split_data = {
    "train": train_records,
    "val": val_records,
    "test": test_records
}

os.makedirs(os.path.dirname(config.DATA_SPLIT_JSON), exist_ok=True)

with open(config.DATA_SPLIT_JSON, 'w') as f:
    json.dump(split_data, f, indent=4)

print(f"Successfully saved data splits configuration to {config.DATA_SPLIT_JSON}")

Successfully saved data splits configuration to /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/data_splits.json
